# BioNexus 3 — Bring your own model

`ModelProvider` has two methods. Any OpenAI-compatible endpoint works out of
the box (OpenAI, ollama, vLLM, OpenRouter); otherwise implement the protocol.

In [ ]:
from bionexus.adapters.models.openai_compat import OpenAICompatProvider
from bionexus.core.contracts import Envelope, ToolInvocation

# 1) point at any OpenAI-compatible server (or leave unset for the offline stub)
# provider = OpenAICompatProvider(model="llama3:8b", base_url="http://localhost:11434/v1")
provider = OpenAICompatProvider()          # stub mode: deterministic, offline

inv = ToolInvocation(envelope=Envelope(experiment_branch_id="nb3"),
                     tool="c.fm.generate_molecules")
result = provider.generate(inv, "propose 3 inhibitors of KRAS_G12C", n=3)
print(result.output["text"])

## Or implement the protocol directly

In [ ]:
import json, random
from bionexus.core.contracts import ToolResult

class MyGenerativeChem:
    name = "my-genchem"
    model_version = "mine-v3"        # pinned -> stamped into provenance

    def generate(self, invocation, prompt, n=3, **kw):
        rng = random.Random(0)       # your model call goes here
        text = json.dumps([{"smiles": "C" * rng.randint(2, 8) + "O"} for _ in range(n)])
        return ToolResult(envelope=invocation.envelope.model_copy(),
                          invocation_id=invocation.invocation_id,
                          status="succeeded", output={"text": text},
                          model_version=self.model_version)

    def predict(self, invocation, kind, payload):
        raise NotImplementedError

from bionexus.core.registry import Registry
registry = Registry()
registry.register_model(MyGenerativeChem(), default=True)
print("registered:", registry.model.name, registry.model.model_version)

The workflow engine and tournament consume only the protocol — swap models per campaign, per stage, or per round without touching pipeline code.